# 🚀 Période 2 — Notebook 03 : Entraînement & MLflow

**Objectif** : entraîner plusieurs variantes du modèle Naïve Bayes,
logger chaque run dans MLflow et comparer les résultats dans l'UI.

Pré-requis : avoir exécuté `02_feature_engineering.ipynb` (fichiers CSV dans `data/processed/`).

In [ ]:
import sys, json, pickle
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report
)
from training.models.naive_bayes_model import MovieNaiveBayesRecommender

# Charger les données préparées
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

with open('../data/processed/feature_columns.json') as f:
    FEATURE_COLS = json.load(f)

print(f'X_train : {X_train.shape}  y_train : {y_train.value_counts().to_dict()}')
print(f'X_test  : {X_test.shape}   y_test  : {y_test.value_counts().to_dict()}')
print(f'Features : {len(FEATURE_COLS)}')

## A. Configurer MLflow (si disponible)

In [ ]:
try:
    import mlflow
    import mlflow.sklearn

    # En local : http://localhost:5000
    # Dans Jupyter via Docker : http://mlflow:5000
    MLFLOW_URI = 'http://mlflow:5000'
    mlflow.set_tracking_uri(MLFLOW_URI)
    mlflow.set_experiment('movielens-recommender')

    print(f'✅ MLflow connecté : {MLFLOW_URI}')
    print(f'   Ouvrir l\'UI : {MLFLOW_URI}')
    MLFLOW_ON = True

except Exception as e:
    print(f'⚠️  MLflow non disponible : {e}')
    print('   Les runs seront entraînés sans tracking.')
    MLFLOW_ON = False

## B. Fonction d'entraînement avec logging MLflow

In [ ]:
def run_experiment(params: dict, run_name: str = None) -> dict:
    """
    Entraîne un modèle avec les params donnés,
    évalue sur le test set et log dans MLflow.
    """
    model = MovieNaiveBayesRecommender(**params)
    model.fit(X_train[FEATURE_COLS], y_train)

    # Cross-validation
    cv = cross_val_score(model, X_train[FEATURE_COLS], y_train,
                         cv=5, scoring='accuracy')

    # Test set
    y_pred  = model.predict(X_test[FEATURE_COLS])
    y_proba = model.predict_proba(X_test[FEATURE_COLS])

    metrics = {
        'accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'f1_score' : round(f1_score(y_test, y_pred, zero_division=0), 4),
        'roc_auc'  : round(roc_auc_score(y_test, y_proba[:,1]), 4),
        'cv_mean'  : round(cv.mean(), 4),
        'cv_std'   : round(cv.std(),  4),
    }

    if MLFLOW_ON:
        with mlflow.start_run(run_name=run_name):
            mlflow.log_params(params)
            mlflow.log_params({'n_features': len(FEATURE_COLS),
                               'n_train': len(X_train),
                               'n_test' : len(X_test)})
            mlflow.log_metrics(metrics)
            mlflow.sklearn.log_model(
                model, 'model',
                registered_model_name='naive-bayes-recommender'
            )

    return {'params': params, 'metrics': metrics, 'model': model}

print('✅ Fonction run_experiment définie')

## C. Run 1 — Modèle de référence (baseline)

In [ ]:
baseline = run_experiment(
    params={
        'gaussian_var_smoothing': 1e-2,
        'bernoulli_alpha':        1.0,
        'gaussian_weight':        0.6,
        'bernoulli_weight':       0.4,
    },
    run_name='baseline'
)

print('📊 Baseline :')
for k, v in baseline['metrics'].items():
    print(f'  {k:12s} : {v:.4f}')

## D. Run 2 — Plus de lissage gaussien

In [ ]:
run2 = run_experiment(
    params={
        'gaussian_var_smoothing': 1e-1,
        'bernoulli_alpha':        1.0,
        'gaussian_weight':        0.6,
        'bernoulli_weight':       0.4,
    },
    run_name='high_smoothing'
)
print('📊 High smoothing :')
for k, v in run2['metrics'].items():
    print(f'  {k:12s} : {v:.4f}')

## E. Run 3 — Poids inversés (genres dominants)

In [ ]:
run3 = run_experiment(
    params={
        'gaussian_var_smoothing': 1e-2,
        'bernoulli_alpha':        0.5,
        'gaussian_weight':        0.3,
        'bernoulli_weight':       0.7,
    },
    run_name='genre_dominant'
)
print('📊 Genre dominant :')
for k, v in run3['metrics'].items():
    print(f'  {k:12s} : {v:.4f}')

## F. Comparaison des runs

In [ ]:
all_runs = [
    ('baseline',       baseline),
    ('high_smoothing', run2),
    ('genre_dominant', run3),
]

print(f'{"-"*65}')
print(f'{"Run":20s} {"Accuracy":>10} {"F1":>8} {"AUC":>8} {"CV Mean":>10}')
print(f'{"-"*65}')

best_run = None
best_acc = 0
for name, run in all_runs:
    m = run['metrics']
    print(f'{name:20s} {m["accuracy"]:10.4f} {m["f1_score"]:8.4f} '
          f'{m["roc_auc"]:8.4f} {m["cv_mean"]:10.4f}')
    if m['accuracy'] > best_acc:
        best_acc = m['accuracy']
        best_run = (name, run)

print(f'{"-"*65}')
print(f'\n🏆 Meilleur run : {best_run[0]}  (accuracy = {best_acc:.4f})')

## G. Sauvegarder le meilleur modèle

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

best_model   = best_run[1]['model']
best_metrics = best_run[1]['metrics']
best_params  = best_run[1]['params']

bundle = {
    'model':           best_model,
    'feature_columns': FEATURE_COLS,
    'metrics':         best_metrics,
    'params':          best_params,
    'run_name':        best_run[0],
}

with open('../models/naive_bayes_recommender.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print(f'✅ Meilleur modèle sauvegardé dans models/naive_bayes_recommender.pkl')
print(f'   Run    : {best_run[0]}')
print(f'   Metrics: {best_metrics}')
print(f'\n→ Passer au notebook 04_model_evaluation.ipynb')